<a href="https://colab.research.google.com/github/iamshekhark/BIS-LAB/blob/main/PSO_for_Neural_Network_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generate dataset
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Neural Network parameters
INPUT = 2
HIDDEN = 8
OUTPUT = 1

# PSO parameters
N = 30
ITERATIONS = 200
W = 0.7
C1 = 1.5
C2 = 1.5

# Total weights and biases
DIM = INPUT * HIDDEN + HIDDEN + HIDDEN * OUTPUT + OUTPUT

# Neural Network prediction
def predict(particle, X):
    idx = 0

    w1 = particle[idx:idx + INPUT * HIDDEN].reshape(INPUT, HIDDEN)
    idx += INPUT * HIDDEN

    b1 = particle[idx:idx + HIDDEN]
    idx += HIDDEN

    w2 = particle[idx:idx + HIDDEN].reshape(HIDDEN, OUTPUT)
    idx += HIDDEN

    b2 = particle[idx:idx + OUTPUT]

    h = np.tanh(np.dot(X, w1) + b1)
    z = np.dot(h, w2) + b2

    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))


# Fitness function
def fitness(particle):
    output = predict(particle, X_train)
    loss = -np.mean(
        y_train.reshape(-1, 1) * np.log(output + 1e-8) +
        (1 - y_train.reshape(-1, 1)) * np.log(1 - output + 1e-8)
    )
    return loss


# Initialize particles
positions = np.random.uniform(-1, 1, (N, DIM))
velocities = np.random.uniform(-0.1, 0.1, (N, DIM))

pbest = positions.copy()
pbest_values = np.array([fitness(p) for p in positions])

best_index = np.argmin(pbest_values)
gbest = pbest[best_index].copy()
gbest_value = pbest_values[best_index]

# PSO main loop
for iteration in range(ITERATIONS):

    for i in range(N):

        r1 = np.random.rand(DIM)
        r2 = np.random.rand(DIM)

        # Update velocity
        velocities[i] = (
            W * velocities[i]
            + C1 * r1 * (pbest[i] - positions[i])
            + C2 * r2 * (gbest - positions[i])
        )

        # Update position
        positions[i] += velocities[i]

        # Calculate fitness
        current = fitness(positions[i])

        # Update personal best
        if current < pbest_values[i]:
            pbest[i] = positions[i].copy()
            pbest_values[i] = current

        # Update global best
        if current < gbest_value:
            gbest = positions[i].copy()
            gbest_value = current

    if (iteration + 1) % 20 == 0:
        print("Iteration:", iteration + 1,
              "Best Loss:", round(gbest_value, 5))

# Final evaluation
train_pred = (predict(gbest, X_train) >= 0.5).astype(int)
test_pred = (predict(gbest, X_test) >= 0.5).astype(int)

train_accuracy = np.mean(
    train_pred.flatten() == y_train
) * 100

test_accuracy = np.mean(
    test_pred.flatten() == y_test
) * 100

print("\nPSO Neural Network Optimization")
print("--------------------------------")
print("Best Fitness (Loss):", round(gbest_value, 5))
print("Training Accuracy:", round(train_accuracy, 2), "%")
print("Testing Accuracy:", round(test_accuracy, 2), "%")


Iteration: 20 Best Loss: 0.06912
Iteration: 40 Best Loss: 0.02213
Iteration: 60 Best Loss: 0.01336
Iteration: 80 Best Loss: 0.00452
Iteration: 100 Best Loss: 0.00245
Iteration: 120 Best Loss: 0.00047
Iteration: 140 Best Loss: 0.00016
Iteration: 160 Best Loss: 0.00011
Iteration: 180 Best Loss: 6e-05
Iteration: 200 Best Loss: 4e-05

PSO Neural Network Optimization
--------------------------------
Best Fitness (Loss): 4e-05
Training Accuracy: 100.0 %
Testing Accuracy: 97.5 %
